# Train v5-1 Feature Engineering + Multi-Seed Ensemble

**Changes from v5**:
- delete log time index


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))


In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import gc
from src.score import weighted_rmse_score as score
import time


## Config

In [ ]:
TRAIN_PATH = PROJECT_ROOT / "data" / "train.parquet"
TEST_PATH  = PROJECT_ROOT / "data" / "test.parquet"
MODEL_DIR  = PROJECT_ROOT / "models"
SUB_DIR    = PROJECT_ROOT / "submissions"

HORIZONS = [1, 3, 10, 25]
SEEDS    = [2]
VERSION = "v5_1"
ZERO_CODES = ['MLAAMU3K', 'EP12UF2K', '1HEMHZK2', 'SJZP0OVU', '83EG83KQ']

SPLIT_INDEX = 3500

# Per-horizon top features for industry-neutral normalization
NEUTRAL_FEATURES = {
    1:  ['feature_al', 'feature_s',  'feature_ce', 'feature_by',
         'feature_y',  'feature_t',  'feature_cf', 'feature_x', 'feature_cb'],
    3:  ['feature_ce', 'feature_cg', 'feature_v',  'feature_cf',
         'feature_y',  'feature_bg', 'feature_al', 'feature_aq', 'feature_z'],
    10: ['feature_v',  'feature_cg', 'feature_ce', 'feature_bg',
         'feature_cf', 'feature_u',  'feature_l',  'feature_z', 'feature_al'],
    25: ['feature_v',  'feature_bg', 'feature_l',  'feature_cg',
         'feature_cf', 'feature_m',  'feature_ce', 'feature_z', 'feature_aq'],
}

BASE_PARAMS = {
    'objective': 'regression',
    'boosting_type': 'gbdt',
    'metric': 'rmse',
    'num_leaves': 80,
    'min_child_samples': 200,
    'lambda_l2': 10,
    'learning_rate': 0.01,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'feature_fraction': 0.8,
    'verbosity': -1,
}

NUM_BOOST_ROUND = 4200
EARLY_STOPPING  = 200

print(f'Horizons: {HORIZONS}')
print(f'Seeds:    {SEEDS}')
print(f'Split:    ts_index <= {SPLIT_INDEX} (train) | > {SPLIT_INDEX} (val)')


Horizons: [1, 3, 10, 25]
Seeds:    [2]
Split:    ts_index <= 3500 (train) | > 3500 (val)


## Feature Engineering Functions

In [4]:
def add_time_features(df):
    """Add time-index derived features."""
    ts = df['ts_index']
    df['ts_mod_7']   = ts % 5
    df['ts_mod_30']  = ts % 21
    df['ts_mod_90']  = ts % 63
    df['ts_sin']     = np.sin(2 * np.pi * ts / 252)
    df['ts_cos']     = np.cos(2 * np.pi * ts / 252)
    df['ts_sin_100'] = np.sin(2 * np.pi * ts / 100)
    df['ts_cos_100'] = np.cos(2 * np.pi * ts / 100)
    return df


def add_industry_neutral(df, horizon):
    """Add industry-neutral features: normalize by code and by sub_category, both per ts_index."""
    feats = NEUTRAL_FEATURES[horizon]
    
    for group_col, suffix in [('code', '_neut_code'), ('sub_category', '_neut_subcat')]:
        # Compute group mean and std per (group_col, ts_index)
        group_stats = df.groupby([group_col, 'ts_index'])[feats].agg(['mean', 'std'])
        # Flatten multi-level columns: (feature, mean) -> feature_mean, etc.
        group_stats.columns = [f'{f}_{s}' for f, s in group_stats.columns]
        group_stats = group_stats.reset_index()
        
        # Merge back
        df = df.merge(group_stats, on=[group_col, 'ts_index'], how='left')
        
        # Compute normalized features
        for f in feats:
            mean_col = f'{f}_mean'
            std_col  = f'{f}_std'
            new_col  = f'{f}{suffix}'
            df[new_col] = (df[f] - df[mean_col]) / (df[std_col] + 1e-8)
            # Drop temp columns
            df.drop(columns=[mean_col, std_col], inplace=True)
    
    return df


def create_features(df, horizon):
    """Full feature engineering pipeline for a single horizon."""
    df = add_time_features(df)
    df = add_industry_neutral(df, horizon)
    return df


def set_cat(df, cat_features):
    for feat in cat_features:
        if feat in df.columns:
            df[feat] = df[feat].astype('category')
    return df


print('Feature engineering functions defined.')


Feature engineering functions defined.


## Train: Per-Horizon × Multi-Seed

For each horizon:
1. Load & filter to horizon
2. Create features (time + industry-neutral)
3. Split train/val on ts_index
4. Train 1 seeds

In [ ]:
CAT_FEATURES = ['code', 'sub_category', 'horizon']
EXCLUDE_COLS = ['id', 'sub_code', 'ts_index', 'weight', 'y_target']

# Storage for all models and val predictions
all_models = {}       # (horizon, seed) -> booster
all_importances = {}  # horizon -> importance DataFrame (averaged across seeds)
val_preds_all = {}    # horizon -> DataFrame with index, per-seed preds, and ensemble
val_meta_all  = {}    # horizon -> (y_target, weight, code) for scoring

for h in HORIZONS:
    t0 = time.time()
    print(f"\n{'='*70}")
    print(f"  HORIZON = {h}")
    print(f"{'='*70}")

    # ── 1. Load & engineer features ──
    print(f"  Loading and engineering features...")
    df_h = create_features(
        pd.read_parquet(TRAIN_PATH).query(f"horizon == {h}").reset_index(drop=True),
        horizon=h
    )

    # ── 2. Identify feature columns ──
    features = [c for c in df_h.columns if c not in EXCLUDE_COLS]
    df_h = set_cat(df_h, CAT_FEATURES)
    print(f"  Total features: {len(features)}")

    # ── 3. Train/Val split ──
    train_mask = df_h['ts_index'] <= SPLIT_INDEX
    val_mask   = ~train_mask
    h_train = df_h[train_mask]
    h_val   = df_h[val_mask]
    print(f"  Train: {len(h_train):,} | Val: {len(h_val):,}")

    # Store val metadata for scoring
    val_meta_all[h] = {
        'y_target': h_val['y_target'].values,
        'weight':   h_val['weight'].values,
        'code':     h_val['code'].values,
        'index':    h_val.index.values,
    }

    # ── 4. LightGBM Datasets (shared across seeds) ──
    dtrain = lgb.Dataset(
        h_train[features], label=h_train['y_target'], weight=h_train['weight'],
        categorical_feature=CAT_FEATURES, free_raw_data=False
    )
    dval = lgb.Dataset(
        h_val[features], label=h_val['y_target'], weight=h_val['weight'],
        categorical_feature=CAT_FEATURES, reference=dtrain, free_raw_data=False
    )
    del h_train, df_h
    gc.collect()
    # ── 5. Multi-seed training ──
    seed_importances = []
    seed_val_preds   = {}

    for seed in SEEDS:
        params = {**BASE_PARAMS, 'seed': seed}
        print(f"\n  --- Seed {seed} ---")

        model = lgb.train(
            params, dtrain,
            valid_sets=[dtrain, dval],
            valid_names=['train', 'val'],
            num_boost_round=NUM_BOOST_ROUND,
            callbacks=[
                lgb.early_stopping(stopping_rounds=EARLY_STOPPING),
                lgb.log_evaluation(period=300)
            ]
        )
        all_models[(h, seed)] = model

        # Save model
        model_path = MODEL_DIR / f"lgb_model_{VERSION}_horizon{h}_seed{seed}.txt"
        model.save_model(model_path, num_iteration=model.best_iteration)
        print(f"    Saved -> {model_path}  (best iter: {model.best_iteration})")

        # Val predictions
        preds = model.predict(h_val[features], num_iteration=model.best_iteration)
        seed_val_preds[seed] = preds

        # Feature importance
        imp = pd.Series(
            model.feature_importance(importance_type='gain'),
            index=features
        )
        seed_importances.append(imp)

    # ── 6. Ensemble val predictions (average across seeds) ──
    val_preds_df = pd.DataFrame(seed_val_preds)
    val_preds_df['ensemble'] = val_preds_df.mean(axis=1)
    val_preds_all[h] = val_preds_df

    # ── 7. Average feature importance ──
    avg_imp = pd.concat(seed_importances, axis=1).mean(axis=1).sort_values(ascending=False)
    all_importances[h] = avg_imp
    print(f"\n  Top-10 features (horizon={h}, averaged over {len(SEEDS)} seeds):")
    print(avg_imp.head(10).to_frame('importance').to_string())

    # ── 8. Free memory ──
    del h_val, dtrain, dval
    gc.collect()
    print(f"\n  Horizon {h} done in {time.time()-t0:.1f}s")



  HORIZON = 1
  Loading and engineering features...
  Total features: 114
  Train: 1,351,193 | Val: 43,460

  --- Seed 2 ---
Training until validation scores don't improve for 200 rounds
[300]	train's rmse: 0.000965807	val's rmse: 0.00113537
[600]	train's rmse: 0.000955592	val's rmse: 0.00113524
Early stopping, best iteration is:
[565]	train's rmse: 0.000956566	val's rmse: 0.00113506
    Saved -> /home/lingod/kaggle_projects/ts_forecasting/models/lgb_v5_1_h1_seed2.txt  (best iter: 565)

  Top-10 features (horizon=1, averaged over 1 seeds):
                          importance
feature_al              2.167450e+06
sub_category            1.982564e+06
feature_by              1.346777e+06
ts_cos_100              1.189043e+06
feature_al_neut_subcat  1.158586e+06
feature_s               1.153145e+06
feature_y               9.646601e+05
ts_cos                  9.279130e+05
feature_cf_neut_subcat  9.175938e+05
feature_by_neut_subcat  8.810095e+05

  Horizon 1 done in 222.6s

  HORIZON = 3
  L

## Validation Scoring
Per-seed, per-horizon, and overall ensemble scores.

In [7]:
print('='*70)
print('  VALIDATION SCORES')
print('='*70)

# ── Per-seed per-horizon scores ──
for h in HORIZONS:
    meta = val_meta_all[h]
    y_true = meta['y_target']
    w      = meta['weight']
    codes  = meta['code']
    zero_mask = np.isin(codes, ZERO_CODES)

    print(f"\n  Horizon {h:>2d}  (zero-coded rows: {zero_mask.sum():,})")
    for seed in SEEDS:
        preds = val_preds_all[h][seed].values.copy()
        preds[zero_mask] = 0.0
        s = score(y_true, preds, w)
        print(f"    Seed {seed:>5d}: {s:.4f}")

    # Ensemble
    ens_preds = val_preds_all[h]['ensemble'].values.copy()
    ens_preds[zero_mask] = 0.0
    ens_score = score(y_true, ens_preds, w)
    print(f"    Ensemble:  {ens_score:.4f}")

# ── Overall ensemble score ──
print(f"\n{'='*70}")
all_y = np.concatenate([val_meta_all[h]['y_target'] for h in HORIZONS])
all_w = np.concatenate([val_meta_all[h]['weight'] for h in HORIZONS])
all_p = []
for h in HORIZONS:
    meta = val_meta_all[h]
    ens = val_preds_all[h]['ensemble'].values.copy()
    zero_mask = np.isin(meta['code'], ZERO_CODES)
    ens[zero_mask] = 0.0
    all_p.append(ens)
all_p = np.concatenate(all_p)

overall = score(all_y, all_p, all_w)
print(f"  Overall Ensemble Val Score: {overall:.4f}")
print('='*70)


  VALIDATION SCORES

  Horizon  1  (zero-coded rows: 7,675)
    Seed     2: 0.0512
    Ensemble:  0.0512

  Horizon  3  (zero-coded rows: 7,618)
    Seed     2: 0.1240
    Ensemble:  0.1240

  Horizon 10  (zero-coded rows: 7,238)
    Seed     2: 0.2065
    Ensemble:  0.2065

  Horizon 25  (zero-coded rows: 6,609)
    Seed     2: 0.2495
    Ensemble:  0.2495

  Overall Ensemble Val Score: 0.2157


## Test Predictions
For each horizon: load test, create features, predict with all seeds, average.

In [8]:
test_preds_parts = []  # list of DataFrames with (id, prediction)

for h in HORIZONS:
    t0 = time.time()
    print(f"\n  Predicting horizon {h}...")

    # Load & engineer
    df_test_h = create_features(
        pd.read_parquet(TEST_PATH).query(f"horizon == {h}").reset_index(drop=True),
        horizon=h
    )
    features = [c for c in df_test_h.columns if c not in EXCLUDE_COLS]
    df_test_h = set_cat(df_test_h, CAT_FEATURES)

    # Predict with each seed
    preds_accum = np.zeros(len(df_test_h), dtype=np.float64)
    for seed in SEEDS:
        model = all_models[(h, seed)]
        preds_accum += model.predict(df_test_h[features], num_iteration=model.best_iteration)
    preds_accum /= len(SEEDS)

    # Zero-code override
    zero_mask = df_test_h['code'].isin(ZERO_CODES)
    preds_accum[zero_mask] = 0.0

    test_preds_parts.append(pd.DataFrame({
        'id': df_test_h['id'].values,
        'prediction': preds_accum
    }))

    print(f"    Horizon {h:>2d}: {len(df_test_h):,} rows, "
          f"zero-coded: {zero_mask.sum():,}  ({time.time()-t0:.1f}s)")

    del df_test_h
    gc.collect()

# Combine
prediction_df = pd.concat(test_preds_parts, ignore_index=True)
assert prediction_df['prediction'].isna().sum() == 0, 'NaN in predictions!'
print(f"\n  Total test predictions: {len(prediction_df):,}")
prediction_df.head()



  Predicting horizon 1...
    Horizon  1: 379,617 rows, zero-coded: 69,005  (6.6s)

  Predicting horizon 3...
    Horizon  3: 376,558 rows, zero-coded: 68,523  (51.9s)

  Predicting horizon 10...
    Horizon 10: 362,057 rows, zero-coded: 65,675  (21.3s)

  Predicting horizon 25...
    Horizon 25: 328,875 rows, zero-coded: 59,033  (27.5s)

  Total test predictions: 1,447,107


,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.006958
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3648,0.000439
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3649,-0.006980
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3650,-0.011073
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3651,-0.008981


## Save Submission

In [9]:
sub_path = f"{SUB_DIR}/submission_v5_1.csv"
prediction_df.to_csv(sub_path, index=False)
print(f"Submission saved -> {sub_path}")
prediction_df.info()
prediction_df.head()


Submission saved -> /home/lingod/kaggle_projects/ts_forecasting/submissions/submission_v5_1.csv
<class 'pandas.DataFrame'>
RangeIndex: 1447107 entries, 0 to 1447106
Data columns (total 2 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   id          1447107 non-null  str    
 1   prediction  1447107 non-null  float64
dtypes: float64(1), str(1)
memory usage: 74.0 MB


,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.006958
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3648,0.000439
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3649,-0.006980
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3650,-0.011073
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3651,-0.008981


## (Optional) Reload Models from Disk

In [ ]:
# all_models = {}
# for h in HORIZONS:
#     for seed in SEEDS:
#         path = f"{MODEL_DIR}/lgb_v5_h{h}_seed{seed}.txt"
#         all_models[(h, seed)] = lgb.Booster(model_file=path)
# print(f'Reloaded {len(all_models)} models.')
